## LAB 01 — Ingestão SupportIQ (challenge 002)

Lê `dados/customer_support_tickets.csv` e grava Delta. Colunas **somente as do CSV** (espaços inclusos). Dataset 2 (`Document` + `Topic_group`) **não está** neste pacote.

### Controle de Alteração de Versões

| versão | data | autor | e-mail | alterações |
| --- | --- | --- | --- | --- |
| 1.2 | 21-SET-2026 | Adriano Aguiar | — | Troca para `customer_support_tickets.csv` do challenge 002 |

### Contrato do CSV (contado no disco 2026-09-21)

| Arquivo | Linhas | Colunas |
| --- | ---: | --- |
| `customer_support_tickets.csv` | 8469 | `Ticket ID`, `Customer Name`, `Customer Email`, `Customer Age`, `Customer Gender`, `Product Purchased`, `Date of Purchase`, `Ticket Type`, `Ticket Subject`, `Ticket Description`, `Ticket Status`, `Resolution`, `Ticket Priority`, `Ticket Channel`, `First Response Time`, `Time to Resolution`, `Customer Satisfaction Rating` |

Grain = ticket. Sem FK.

Armadilhas do dump (CSV ganha o brief):

- Brief cita ~30 000 linhas; disco = **8469**.
- `Ticket Description` / `Resolution` têm **newline** — Spark precisa de `multiLine=true` (sem isso lê ~18762 linhas).
- Nomes com espaço (`Ticket ID`) exigem Delta column mapping (`name`) no Unity Catalog.
- `Ticket Type` tem **5** valores (Refund, Technical, Cancellation, Product inquiry, Billing) — não só 3.
- `Time to Resolution` e `First Response Time` são **datetime**, não horas. Duração = TTR − FRT, só em Closed.
- Closed ⇔ `Resolution` + TTR + CSAT preenchidos (2769). Open ⇔ FRT/TTR/Resolution/CSAT nulos. Pending ⇔ tem FRT, sem TTR/CSAT.
- Dataset 2 (IT classification, ~48k) **ausente**. Não inventar `Topic_group`.
- `Customer Name` / `Customer Email` são PII — não dump no PR.

### Como executar no Databricks

1. Importe a pasta `notebook-process-002-support` para o Workspace (incluindo `dados/`).
2. Widgets: `catalog_name` (default `workspace`), `schema_name` (`supportiq`), `data_path` (vazio = auto).
3. `data_path` se o auto-detect falhar: `/Workspace/Users/<voce>/notebook-process-002-support/dados`.
4. Cluster ou serverless → **Run All**.
5. Lab 02 e Lab 04 leem `workspace.supportiq.customer_support_tickets`.
6. SQL: colunas com espaço exigem backtick — `` `Ticket ID` ``.

In [ ]:
from pathlib import Path

EXPECTED = {
    "customer_support_tickets": {
        "rows": 8469,
        "columns": [
            "Ticket ID",
            "Customer Name",
            "Customer Email",
            "Customer Age",
            "Customer Gender",
            "Product Purchased",
            "Date of Purchase",
            "Ticket Type",
            "Ticket Subject",
            "Ticket Description",
            "Ticket Status",
            "Resolution",
            "Ticket Priority",
            "Ticket Channel",
            "First Response Time",
            "Time to Resolution",
            "Customer Satisfaction Rating",
        ],
    },
}

INT_COLUMNS = {"Ticket ID", "Customer Age"}
DOUBLE_COLUMNS = {"Customer Satisfaction Rating"}
DATE_COLUMNS = {"Date of Purchase"}
TIMESTAMP_COLUMNS = {"First Response Time", "Time to Resolution"}


def widget_or_default(name: str, default: str) -> str:
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


def resolve_data_path(explicit: str) -> str:
    marker = "customer_support_tickets.csv"
    candidates = []
    if explicit:
        candidates.append(Path(explicit))
    try:
        notebook_path = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        parts = [part for part in notebook_path.split("/") if part]
        if "notebook-process-002-support" in parts:
            idx = parts.index("notebook-process-002-support")
            repo = "/".join(parts[: idx + 1])
            candidates.append(Path("/Workspace") / repo / "dados")
            candidates.append(Path("/") / repo / "dados")
    except Exception:
        pass
    here = Path.cwd()
    candidates.extend(
        [
            here / "dados",
            here.parent / "dados",
            here.parent.parent / "dados",
        ]
    )
    for candidate in candidates:
        if (candidate / marker).exists():
            return str(candidate)
    raise FileNotFoundError(
        "Não achei customer_support_tickets.csv. Defina o widget data_path "
        "para a pasta dados/ no Workspace ou em um Volume."
    )


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "supportiq")
data_path = resolve_data_path(widget_or_default("data_path", ""))
print(f"catalog={catalog_name} schema={schema_name}")
print(f"data_path={data_path}")

In [ ]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"USE SCHEMA {schema_name}")

In [ ]:
from pyspark.sql import functions as F

loaded = {}
for entity_name, contract in EXPECTED.items():
    csv_path = str(Path(data_path) / f"{entity_name}.csv")
    table_name = f"{catalog_name}.{schema_name}.{entity_name}"
    sdf = (
        spark.read.option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "true")
        .option("quote", "\"")
        .option("escape", "\"")
        .option("nullValue", "")
        .option("mode", "PERMISSIVE")
        .csv(csv_path)
    )
    missing = [col for col in contract["columns"] if col not in sdf.columns]
    extra = [col for col in sdf.columns if col not in contract["columns"]]
    if missing or extra:
        raise ValueError(
            f"{entity_name}: colunas fora do CSV. missing={missing} extra={extra}"
        )
    sdf = sdf.select(*[F.col(f"`{col}`") for col in contract["columns"]])
    for column in contract["columns"]:
        quoted = f"`{column}`"
        if column in INT_COLUMNS:
            sdf = sdf.withColumn(column, F.col(quoted).cast("int"))
        elif column in DOUBLE_COLUMNS:
            sdf = sdf.withColumn(column, F.col(quoted).cast("double"))
        elif column in DATE_COLUMNS:
            sdf = sdf.withColumn(column, F.to_date(F.col(quoted).cast("string")))
        elif column in TIMESTAMP_COLUMNS:
            sdf = sdf.withColumn(column, F.to_timestamp(F.col(quoted).cast("string")))
    row_count = sdf.count()
    if row_count != contract["rows"]:
        raise ValueError(
            f"{entity_name}: esperado {contract['rows']} linhas, lido {row_count}"
        )
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")
    (
        sdf.write.format("delta")
        .mode("overwrite")
        .option("delta.columnMapping.mode", "name")
        .option("delta.minReaderVersion", "2")
        .option("delta.minWriterVersion", "5")
        .saveAsTable(table_name)
    )
    loaded[entity_name] = {"rows": row_count, "cols": sdf.columns}
    print(f"{table_name} rows={row_count}")

loaded

In [ ]:
display(spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}"))

In [ ]:
display(
    spark.table(f"{catalog_name}.{schema_name}.customer_support_tickets").limit(5)
)